In [ ]:
# 09 - Final Prediction System

# Trains the final tuned Decision Tree models on the data, saves them to `../Models/`, and provides a `predict_game()` function that takes Pclatform, Genre, Publisher, Year and returns:
# 1. Predicted sales category (Flop / Average / Hit) with probabilities
# 2. Predicted global sales in millions 

In [ ]:
import os, joblib
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import accuracy_score, f1_score, r2_score

df = pd.read_csv("feature_engineered_dataset.csv")
class_names = ['Flop', 'Average', 'Hit']

In [ ]:
 # 1. Build features (publisher grouping + one-hot encoding) -->

In [ ]:
top_publishers = list(df['Publisher'].value_counts().nlargest(15).index)
df['Publisher_Grouped'] = df['Publisher'].where(df['Publisher'].isin(top_publishers), 'Other')

X = pd.get_dummies(df[['Platform', 'Year', 'Genre', 'Publisher_Grouped']],
                   columns=['Platform', 'Genre', 'Publisher_Grouped'], drop_first=True)
y_class = df['Sales_Category'].map({'Flop': 0, 'Average': 1, 'Hit': 2})
y_reg = df['Global_Sales']

X_train, X_test, yc_train, yc_test, yr_train, yr_test = train_test_split(
    X, y_class, y_reg, test_size=0.2, random_state=42)

In [ ]:
#  2. Train final models
# - Category (Flop/Average/Hit): tuned Decision Tree Classifier (parameters from notebook 08) - chosen for interpretability and class probabilities.
# - Sales amount: we compare a Decision Tree Regressor with Linear Regression on the test set and keep the better one.

# First evaluate on the held-out test set, then refit on all data for deployment (standard practice once hyper-parameters are chosen).

In [ ]:
BEST_CLF = dict(criterion='gini', max_depth=15, min_samples_leaf=5, random_state=42)   # from GridSearchCV in nb 08
BEST_REG = dict(max_depth=10, min_samples_leaf=20, random_state=42)   # tree regressor (for comparison)

# --- Honest evaluation on held-out test data ---
clf_eval = DecisionTreeClassifier(**BEST_CLF).fit(X_train, yc_train)
p = clf_eval.predict(X_test)
print(f"Decision Tree Classifier -> Accuracy: {accuracy_score(yc_test, p):.4f} | Macro F1: {f1_score(yc_test, p, average='macro'):.4f}")

# Sales-amount model: compare Decision Tree Regressor vs Linear Regression, keep the better one
tree_reg = DecisionTreeRegressor(**BEST_REG).fit(X_train, np.log1p(yr_train))
lin_reg  = LinearRegression().fit(X_train, np.log1p(yr_train))
r2_tree = r2_score(np.log1p(yr_test), tree_reg.predict(X_test))
r2_lin  = r2_score(np.log1p(yr_test), lin_reg.predict(X_test))
print(f"Tree Regressor   R2 (log scale): {r2_tree:.4f}")
print(f"Linear Regression R2 (log scale): {r2_lin:.4f}")
REGRESSOR_CLASS = LinearRegression if r2_lin >= r2_tree else DecisionTreeRegressor
print("Chosen for sales amount:", REGRESSOR_CLASS.__name__)

# --- Refit on ALL data for the deployed model ---
clf_final = DecisionTreeClassifier(**BEST_CLF).fit(X, y_class)
reg_final = (LinearRegression() if REGRESSOR_CLASS is LinearRegression
             else DecisionTreeRegressor(**BEST_REG)).fit(X, np.log1p(y_reg))

In [ ]:
# 3. Save everything needed for prediction

In [ ]:
os.makedirs("../Models", exist_ok=True)
bundle = {
    'classifier': clf_final,
    'regressor': reg_final,
    'feature_columns': list(X.columns),
    'top_publishers': top_publishers,
    'class_names': class_names,
    'known_platforms': sorted(df['Platform'].unique()),
    'known_genres': sorted(df['Genre'].unique()),
    'year_range': (int(df['Year'].min()), int(df['Year'].max())),
}
joblib.dump(bundle, "../Models/video_game_sales_model.joblib")
print("Saved ->", os.path.abspath("../Models/video_game_sales_model.joblib"))

In [ ]:
# 4. The prediction function

In [ ]:
def predict_game(platform, genre, publisher, year, model_path="../Models/video_game_sales_model.joblib"):
    """Predict sales category and global sales (millions) for a new game."""
    b = joblib.load(model_path)

    # Input validation
    if platform not in b['known_platforms']:
        raise ValueError(f"Unknown platform '{platform}'. Choose from: {b['known_platforms']}")
    if genre not in b['known_genres']:
        raise ValueError(f"Unknown genre '{genre}'. Choose from: {b['known_genres']}")

    # Same preprocessing as training
    pub = publisher if publisher in b['top_publishers'] else 'Other'
    row = pd.DataFrame([{'Platform': platform, 'Year': year, 'Genre': genre, 'Publisher_Grouped': pub}])
    row = pd.get_dummies(row, columns=['Platform', 'Genre', 'Publisher_Grouped'])
    row = row.reindex(columns=b['feature_columns'], fill_value=0)   # align columns with training

    cls = int(b['classifier'].predict(row)[0])
    proba = b['classifier'].predict_proba(row)[0]
    sales = max(0.01, float(np.expm1(b['regressor'].predict(row)[0])))   # sales can't be below the dataset minimum (0.01M)

    return {
        'input': {'platform': platform, 'genre': genre, 'publisher': publisher, 'year': year},
        'predicted_category': b['class_names'][cls],
        'probabilities': {n: round(float(pr), 3) for n, pr in zip(b['class_names'], proba)},
        'predicted_global_sales_millions': round(sales, 2),
    }

In [ ]:
# 5. Try it

In [ ]:
tests = [
    ("PS2",  "Action",  "Electronic Arts", 2005),
    ("Wii",  "Sports",  "Nintendo",        2008),
    ("PC",   "Strategy","Some Indie Studio", 2012),
    ("DS",   "Puzzle",  "Nintendo",        2007),
    ("PS4",  "Shooter", "Activision",      2015),
]
for t in tests:
    r = predict_game(*t)
    print(f"{t[0]:5s} | {t[1]:9s} | {t[2]:18s} | {t[3]} -> {r['predicted_category']:8s} "
          f"| ~{r['predicted_global_sales_millions']:.2f}M | {r['probabilities']}")

In [ ]:
# 6. Limitations 
# Accuracy is about 56-57% on 3 classes. The tree uses only platform, genre, publisher and year - no reviews, budget or marketing data.
# "Hit/Average/Flop" are relative labels (quartiles of this dataset), not profit-based.
# Unseen publishers are mapped to "Other". Platforms or genres not in the data raise an error.
# The dataset ends around 2016, so predictions for newer platforms/years are not reliable.